In [1]:
import pandas as pd
import numpy as np
import optuna
import plotly.express as px

import xgboost as xgb
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [2]:
train_img_dict = np.load("/kaggle/input/property-valuation-03encodings/train_image_features_b4.npy", allow_pickle=True).item()

train_encoding_df = pd.DataFrame.from_dict(train_img_dict, orient="index").reset_index()
train_encoding_df = train_encoding_df.rename(columns={"index":"house_id"})

model_1 = StandardScaler()
model_1.fit(train_encoding_df.drop(columns="house_id"))
std_arr = model_1.transform(train_encoding_df.drop(columns="house_id"))

pca = PCA(n_components=0.40, random_state=42)
pca_arr = pca.fit_transform(std_arr)

pca_df = pd.DataFrame(pca_arr)
pca_df['house_id'] = train_encoding_df['house_id']
pca_df = pca_df.set_index("house_id").reset_index()

In [3]:
df = pd.read_csv("/kaggle/input/property-val-dataset/train.csv")
df['date'] = pd.to_datetime(df['date'])
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.month
df['day_of_month'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek

df['effective_built_year'] = df.apply(lambda x: x['yr_renovated'] if x['yr_renovated'] > 0 else x['yr_built'], axis=1)
df['house_age'] = df['sale_year'] - df['effective_built_year']

zip_map = df.groupby('zipcode').apply(lambda x: x['price'].median()).to_dict()
df['zip_index'] = df['zipcode'].map(zip_map)

/tmp/ipykernel_17/989552771.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  zip_map = df.groupby('zipcode').apply(lambda x: x['price'].median()).to_dict()


In [4]:
final_df = df.merge(pca_df,how='outer',left_on = 'id',right_on='house_id')
cols_to_drop = ['id', 'date','zipcode',"house_id"]  
final_df.drop(columns=cols_to_drop,inplace=True)
final_df

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,2,3,4,5,6,7,8,9,10,11
0,280000,6,3.00,2400,9373,2.0,0,0,3,7,...,0.290877,-6.073594,-5.633253,-13.611466,-14.934451,9.462794,8.747434,-0.249940,1.001352,-2.436376
1,647500,4,1.75,2060,26036,1.0,0,0,4,8,...,-11.211393,-9.193542,-0.768384,6.044438,-13.556029,7.010575,-9.766714,-0.438059,-1.912871,-4.622661
2,400000,3,1.00,1460,43000,1.0,0,0,3,7,...,9.438823,13.007943,-4.501375,4.451370,-7.879398,0.438834,-3.210675,-1.392112,-2.053820,7.319651
3,235000,3,1.00,1430,7599,1.5,0,0,4,6,...,6.559885,7.643277,-4.579039,11.569905,-9.362732,-2.524021,-0.801762,-0.607516,3.068925,-1.173321
4,402500,4,2.00,1650,3504,1.0,0,0,3,7,...,-1.523940,-2.860825,9.500791,6.350972,6.052811,-2.354280,-2.203408,7.359640,1.067262,-0.223134
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16204,998500,2,1.00,1570,4400,1.5,0,0,4,8,...,2.657553,-3.777408,1.196425,13.551541,10.220032,-7.813064,2.970876,4.858222,-6.999898,-2.546487
16205,380000,2,1.00,1040,7372,1.0,0,0,5,7,...,-0.298760,15.197111,8.958596,0.699172,13.232842,5.152641,-1.052419,-0.207936,-6.360192,3.300368
16206,339000,3,1.00,1100,4128,1.0,0,0,4,7,...,1.878078,11.144259,5.100299,-2.692226,9.648942,7.732147,-4.331114,-5.594069,1.839295,5.805600
16207,399900,2,1.75,1410,1005,1.5,0,0,3,9,...,3.075870,-16.233587,-7.600594,-12.394133,-6.577404,12.947015,12.007878,-4.870407,-7.352153,-5.045096


In [5]:
X = final_df.drop(columns='price')
y = final_df['price']

# Convert once
dtrain = xgb.DMatrix(X.values, label=y.values)

In [6]:
def objective(trial):
    params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'eval_metric': 'rmse',
        'verbosity': 0,
        'nthread': -1,  # while using GPU remove it

        'n_estimators': 5000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),

        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        'max_depth': trial.suggest_int('max_depth', 3, 12),

        'subsample': trial.suggest_float('subsample', 0.85, 1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.85, 1),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 20.0, log=True),

        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),

        #'max_bin': trial.suggest_categorical('max_bin', [256, 512]),
        'max_bin' : 512
    }

    if params['grow_policy'] == 'lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 32, 512)

    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=params['n_estimators'],
        nfold=5,
        early_stopping_rounds=75,
        seed=42,
        verbose_eval=False
    )

    return cv_results['test-rmse-mean'].min()

# Run the detailed study
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=180) # Need more trials for this larger space

print("Best RMSE:", study.best_value)
print("Best Params:", study.best_params)

[I 2026-01-05 14:48:03,298] A new study created in memory with name: no-name-9299e7ac-0918-41c3-a7ad-4ef21234a404
[I 2026-01-05 14:50:47,784] Trial 0 finished with value: 120187.27556946412 and parameters: {'learning_rate': 0.014267908662627804, 'grow_policy': 'depthwise', 'max_depth': 10, 'subsample': 0.8935569720278643, 'colsample_bytree': 0.9220602181921714, 'reg_alpha': 0.08707943662065011, 'reg_lambda': 0.022859036413742703, 'min_child_weight': 6, 'gamma': 4.090504151796737e-05}. Best is trial 0 with value: 120187.27556946412.
[I 2026-01-05 14:51:10,307] Trial 1 finished with value: 118961.63996016703 and parameters: {'learning_rate': 0.07903497969232565, 'grow_policy': 'depthwise', 'max_depth': 7, 'subsample': 0.891989011706407, 'colsample_bytree': 0.8980193829561898, 'reg_alpha': 2.367650658203769e-05, 'reg_lambda': 3.80095241584031e-05, 'min_child_weight': 7, 'gamma': 0.013672323763905807}. Best is trial 1 with value: 118961.63996016703.
[I 2026-01-05 14:53:54,801] Trial 2 fini

Best RMSE: 111736.39825640451
Best Params: {'learning_rate': 0.051179166440270964, 'grow_policy': 'depthwise', 'max_depth': 3, 'subsample': 0.9236126590824028, 'colsample_bytree': 0.8756451342494557, 'reg_alpha': 19.11532865442273, 'reg_lambda': 0.0006386455580355257, 'min_child_weight': 6, 'gamma': 0.011557488335842095}


In [7]:
# 1. Extract the best hyperparameters from the study
best_params = study.best_params

# 2. Add the necessary fixed parameters
# We ensure GPU is enabled and the objective is set correctly
best_params['objective'] = 'reg:squarederror'
best_params['tree_method'] = 'auto' # gpu_hist
best_params['eval_metric'] = 'rmse'

if 'n_estimators' not in best_params:
    best_params['n_estimators'] = 2500

final_model = xgb.XGBRegressor(**best_params)

def eval(y_test,y_pred) :
  print("MSE : ",mean_squared_error(y_test,y_pred))
  print("RMSE : ",mean_squared_error(y_test,y_pred)**0.5)
  print("R2 : ",r2_score(y_test,y_pred))
  return

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
eval(y_test,y_pred)

MSE :  17498294272.0
RMSE :  132281.11835027704
R2 :  0.8838286995887756


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
eval(y_test,y_pred)

MSE :  15095598080.0
RMSE :  122864.14481043686
R2 :  0.8994624018669128


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=99)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
eval(y_test,y_pred)

MSE :  14888591360.0
RMSE :  122018.81559825108
R2 :  0.8825863599777222


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
eval(y_test,y_pred)

MSE :  11730566144.0
RMSE :  108307.73815383646
R2 :  0.9104171395301819


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=538)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
eval(y_test,y_pred)

MSE :  14311865344.0
RMSE :  119632.20863964688
R2 :  0.8840804100036621


In [12]:
print(best_params)

{'learning_rate': 0.051179166440270964, 'grow_policy': 'depthwise', 'max_depth': 3, 'subsample': 0.9236126590824028, 'colsample_bytree': 0.8756451342494557, 'reg_alpha': 19.11532865442273, 'reg_lambda': 0.0006386455580355257, 'min_child_weight': 6, 'gamma': 0.011557488335842095, 'objective': 'reg:squarederror', 'tree_method': 'auto', 'eval_metric': 'rmse', 'n_estimators': 2500}
